# 12.04 · Preference tuning: DPO & friends

**Chapter:** Fine-tuning & Post-training · **Milestone:** M6

## Goals
- [ ] RLHF vs DPO vs ORPO/KTO/SimPO: what each needs and optimises
- [ ] Build preference pairs with **hard negatives** (near-miss mistakes)
- [ ] Run DPO on top of a weak SFT model and compare with a fair control (more SFT on the same data budget)

## Resources
- [DPO paper](https://arxiv.org/abs/2305.18290) · [TRL DPOTrainer](https://huggingface.co/docs/trl/dpo_trainer) — free
- [HF smol course – preference alignment](https://github.com/huggingface/smol-course) — free
- [Nathan Lambert – RLHF book](https://rlhfbook.com/) — free

| Method | Data | Notes |
|---|---|---|
| **RLHF (PPO)** | preferences → reward model → RL | the original ChatGPT recipe; complex, unstable |
| **DPO** | (prompt, chosen, rejected) | no reward model, no sampling; a classification-style loss vs a frozen reference (04.04) |
| **ORPO** | same pairs, no reference model | SFT + preference in one stage |
| **KTO** | single examples labelled good/bad | when you only have thumbs up/down (11.05!) |
| **SimPO**, IPO… | pairs | DPO variants with different regularisation |

Preference tuning teaches **"this, not that"**: useful when the failure is a *specific wrong behaviour* the model keeps choosing.

In [ ]:
import json
import random
import time

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer

from aieng import DATA
from task import dataset, messages, score

FT = DATA / "ft"; FT.mkdir(exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
BASE = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(BASE)
test = dataset(80, seed=2)


def generate(model, msgs):
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)["input_ids"].to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=48, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)


def evaluate(model):
    s = [score(generate(model, messages(e["text"])), e["target"]) for e in test]
    return {k: round(sum(x[k] for x in s) / len(s), 2) for k in s[0]}

## 1. A deliberately weak SFT model
Small budget (200 examples, 25 steps) so there's room to improve. In practice: your SFT model plus a list of its recurring mistakes.

In [ ]:
def sft(rows, steps, out):
    model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32).to(device)
    ds = Dataset.from_list([{"prompt": messages(e["text"]), "completion": [{"role": "assistant", "content": json.dumps(e["target"])}]} for e in rows])
    t = SFTTrainer(model=model, processing_class=tok, train_dataset=ds,
                   args=SFTConfig(output_dir=str(out), max_steps=steps, per_device_train_batch_size=8, learning_rate=1e-3,
                                  logging_steps=50, save_strategy="no", report_to="none", max_length=256),
                   peft_config=LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM"))
    t.train()
    return t.model.merge_and_unload().eval()  # merge so DPO starts from a plain model (its own frozen reference)


torch.manual_seed(0)
sft_rows = dataset(200, seed=11)
weak = sft(sft_rows, steps=25, out=FT / "weak-sft")
print("weak SFT:", evaluate(weak))
for e in test[:6]:
    out = generate(weak, messages(e["text"]))
    if not score(out, e["target"])["exact"]:
        print(f"  ✗ {e['text']!r}\n     got {out!r}\n     want {json.dumps(e['target'])}")

## 2. Preference pairs with hard negatives
`chosen` = the correct JSON. `rejected` = a *plausible* mistake — the kind the model actually makes: dates in the input's format instead of ISO,
currency symbols instead of codes, amounts as strings, an off-by-one field, chatty wrappers. Hard negatives teach sharper boundaries than
random garbage would.

In [ ]:
def corrupt(target, rng):
    t = dict(target)
    kind = rng.choice(["date_format", "currency_symbol", "amount_string", "wrong_day", "chatty"])
    if kind == "date_format":
        y, m, d = t["date"].split("-"); t["date"] = f"{d}/{m}/{y}"
    elif kind == "currency_symbol":
        t["currency"] = {"PEN": "S/", "USD": "$", "EUR": "€"}[t["currency"]]
    elif kind == "amount_string":
        t["amount"] = f"{t['amount']} {t['currency']}"
    elif kind == "wrong_day":
        y, m, d = t["date"].split("-"); t["date"] = f"{y}-{m}-{min(int(d) + 1, 28):02d}"
    else:
        return f"Sure! Here is the payment: {json.dumps(t)}"
    return json.dumps(t)


rng = random.Random(0)
pref_rows = dataset(400, seed=12)
pairs = [{"prompt": messages(e["text"]), "chosen": [{"role": "assistant", "content": json.dumps(e["target"])}],
          "rejected": [{"role": "assistant", "content": corrupt(e["target"], rng)}]} for e in pref_rows]
print(json.dumps(pairs[0]["chosen"]), "\n", json.dumps(pairs[0]["rejected"]))

## 3. DPO
`beta` controls how far the policy may move from the reference (0.1 is a common start). With a PEFT config, TRL uses the model *with the adapter
disabled* as the frozen reference — no second copy in memory.

In [ ]:
dpo_args = DPOConfig(output_dir=str(FT / "dpo-run"), max_steps=50, per_device_train_batch_size=8, learning_rate=5e-4, beta=0.1,
                     logging_steps=10, save_strategy="no", report_to="none", max_length=256)
start = time.perf_counter()
dpo_trainer = DPOTrainer(model=weak, args=dpo_args, train_dataset=Dataset.from_list(pairs), processing_class=tok,
                         peft_config=LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM"))
dpo_trainer.train()
print(f"DPO: {time.perf_counter() - start:.0f}s")
for h in dpo_trainer.state.log_history:
    if "loss" in h:
        print(f"  step {h['step']:3}: loss {h['loss']:.3f} | reward margin {h.get('rewards/margins', float('nan')):.2f} | accuracy {h.get('rewards/accuracies', float('nan')):.2f}")
dpo_model = dpo_trainer.model.eval()

`rewards/accuracies` = how often the policy prefers chosen over rejected (relative to the reference); `rewards/margins` = by how much.

## 4. Compare with a fair control
DPO saw 400 extra examples. The honest comparison is **SFT on those same 400 examples' correct answers** (no negatives). If that matches DPO,
the gain came from more data, not from the preference signal.

In [ ]:
torch.manual_seed(0)
control = sft(sft_rows + pref_rows, steps=75, out=FT / "control-sft")
results = {}
with dpo_trainer.model.disable_adapter():  # the DPO adapter off = the original weak SFT model
    results["weak SFT (200 ex)"] = evaluate(dpo_trainer.model)
results["weak SFT + DPO (400 pairs)"] = evaluate(dpo_model)
results["SFT on 600 ex (control)"] = evaluate(control)
for name, r in results.items():
    print(f"{name:28} {r}")

On a narrow task with exact answers, plain SFT on more correct data is usually as good or better — preference tuning shines where "correct" is
hard to write down but easy to compare (tone, helpfulness, safety, choosing between valid answers). That's the real lesson: **match the method
to the signal you have**.

## Exercises
1. Use the weak model's **own** wrong outputs as `rejected` (on-policy negatives) instead of synthetic corruptions. Better?
2. Sweep `beta` (0.05, 0.1, 0.5). How do reward margins and exact-match change?
3. Try `ORPOTrainer` or `KTOTrainer` (KTO: unpaired good/bad labels — like 11.05's thumbs feedback) on the same data.

In [ ]:
# your code here